#### Task 1: Filter transactions from the United Kingdom and count unique CustomerIDs.

In [2]:
import os
import numpy as np
import pandas as pd
 
HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
 
 
def load_data():
    """Load the real OnlineRetail.csv if it is in this folder, otherwise the simulated sample."""
    for name in ("OnlineRetail.csv", "OnlineRetail_sample.csv"):
        path = os.path.join(HERE, name)
        if os.path.exists(path):
            df = pd.read_csv(path, encoding="ISO-8859-1", dtype={"InvoiceNo": str, "StockCode": str})
            print(f"Loaded {name}: {len(df):,} rows" + ("" if name == "OnlineRetail.csv" else "  (SIMULATED STAND-IN DATA, not the real file)"))
            break
    else:
        raise FileNotFoundError("Put OnlineRetail.csv (or run generate_sample_data.py) in the same folder.")
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    df["LineTotal"] = df["Quantity"] * df["UnitPrice"]
    return df
 
 
def valid_rows(df):
    """Drop cancellations (InvoiceNo starts with C) and rows with non-positive quantity or price."""
    return df[~df["InvoiceNo"].str.startswith("C") & (df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
 
 
df = load_data()
 
# Excel equivalent: Data > Filter, Country = "United Kingdom"
uk = df[df["Country"] == "United Kingdom"]
print(f"\nUK transactions (rows): {len(uk):,} of {len(df):,} ({len(uk) / len(df):.1%})")
print(f"UK rows with a missing CustomerID: {uk['CustomerID'].isna().sum():,}")
 
# Unique CustomerIDs (blank IDs are not customers, so they are not counted)
unique_customers = uk["CustomerID"].dropna().nunique()
print(f"\nUnique CustomerIDs who made purchases from the UK = {unique_customers:,}")
 
# Optional stricter view: only customers with at least one real purchase (not just a cancellation)
buyers = valid_rows(uk)["CustomerID"].dropna().nunique()
print(f"Unique UK CustomerIDs with at least one valid purchase invoice = {buyers:,}")

Loaded OnlineRetail_sample.csv: 3,989 rows  (SIMULATED STAND-IN DATA, not the real file)

UK transactions (rows): 3,194 of 3,989 (80.1%)
UK rows with a missing CustomerID: 249

Unique CustomerIDs who made purchases from the UK = 145
Unique UK CustomerIDs with at least one valid purchase invoice = 145
